# Identifying Deepfakes - V33 Dual-Cluster Refinement

## Fixing the One-Class Logic
In V32, we unintentionally threw away the K-Means baseline and attempted to predict the entire dataset using only a localized LOF score. This caused a massive recall drop. 
**V33 fixes this.** We retain the elite 88%+ K-Means baseline and use anomaly detection strictly as a surgical refiner.

## Dual-Cluster "Polish"
We now attack the error from both sides:
1. **False Negative Hunter:** Fits a Cosine LOF on the Real core. Images inside the Real cluster that look "too synthetic" are flipped to **Fake (1)**.
2. **False Positive Hunter:** Fits a Cosine LOF on the Fake core. Images inside the Fake cluster that look "too biological" are flipped to **Real (0)**.

By leveraging the 99% Precision of the Cosine LOF, we ensure that every flip is mathematically robust, pushing our final accuracy beyond the neural baseline.

In [ ]:
import os, warnings, zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_PATH = '/content'; FOLDER_PATH = '/content/drive/MyDrive/DataMining/project_dataset'
else:
    BASE_PATH = './'; FOLDER_PATH = './project_dataset'

REAL_IMAGE_DIR, FAKE_IMAGE_DIR = os.path.join(BASE_PATH, 'Real-img'), os.path.join(BASE_PATH, 'Image')
RESOLUTION, BATCH_SIZE, SEED = 224, 64, 42

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def extract_if_needed(zip_name, target_dir):
    if not os.path.exists(target_dir) and os.path.exists(os.path.join(FOLDER_PATH, zip_name)):
        with zipfile.ZipFile(os.path.join(FOLDER_PATH, zip_name), 'r') as z: z.extractall(BASE_PATH)

extract_if_needed('Real-img.zip', REAL_IMAGE_DIR)
extract_if_needed('Fake-img.zip', FAKE_IMAGE_DIR)

class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.r = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(('.jpg', '.png'))] if os.path.exists(real_dir) else []
        self.f = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(('.jpg', '.png'))] if os.path.exists(fake_dir) else []
        
        # Enforce perfect 50/50 dataset balance
        min_len = min(len(self.r), len(self.f))
        self.r = self.r[:min_len]
        self.f = self.f[:min_len]
        
        self.all_files = self.r + self.f
        self.labels = [0]*len(self.r) + [1]*len(self.f)
        self.transform = transform
    def __len__(self): return len(self.all_files)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.all_files[idx]).convert('RGB')
            if self.transform: img = self.transform(img)
            return img, self.labels[idx], self.all_files[idx]
        except: return torch.zeros((3, RESOLUTION, RESOLUTION)), self.labels[idx], self.all_files[idx]

full_dataset = DeepfakeDataset(REAL_IMAGE_DIR, FAKE_IMAGE_DIR, transform=transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
]))
all_idx = list(range(len(full_dataset)))
np.random.shuffle(all_idx)

NETWORK_POOL_SIZE = int(len(full_dataset) * 0.05)  
network_dataset = Subset(full_dataset, all_idx[:NETWORK_POOL_SIZE])
network_loader = DataLoader(network_dataset, batch_size=BATCH_SIZE, shuffle=True)
ml_loader = DataLoader(Subset(full_dataset, all_idx[NETWORK_POOL_SIZE:]), batch_size=BATCH_SIZE, shuffle=False)
print(f"Total Balanced Images Loaded: {len(full_dataset)}")

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Layer 4 Unfrozen
locked_layers = ['conv1', 'bn1', 'layer1', 'layer2', 'layer3']
for name, param in model.named_parameters():
    if any(layer_name in name for layer_name in locked_layers): param.requires_grad = False
model.fc = nn.Linear(512, 2)
model = model.to(device)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()

if len(network_loader) > 0:
    for epoch in range(1):
        model.train()
        pbar = tqdm(network_loader, desc=f"Epoch 1/1")
        for imgs, labels, _ in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()

In [ ]:
fc_layer = model.fc
model.fc = nn.Identity()
model.eval()

latent_embeddings, weak_predictions, ground_truth = [], [], []

with torch.no_grad():
    for imgs, labels, _ in tqdm(ml_loader, desc="Extraction"):
        imgs = imgs.to(device)
        latent = model(imgs)
        outputs = fc_layer(latent)
        _, preds = outputs.max(1)
        latent_embeddings.append(latent.cpu().numpy())
        weak_predictions.extend(preds.cpu().numpy())
        ground_truth.extend(labels.numpy())

if len(latent_embeddings) > 0:
    latent_embeddings = np.vstack(latent_embeddings)
    weak_predictions = np.array(weak_predictions)
    ground_truth = np.array(ground_truth)

print(f"\n--> Raw ResNet Neural Prediction Baseline Accuracy: {accuracy_score(ground_truth, weak_predictions):.4f}\n")

In [ ]:
print("Phase 1: Establishing Balanced K-Means Baseline")
clusterer = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = clusterer.fit_predict(latent_embeddings)

mean_weak_0 = np.mean(weak_predictions[cluster_labels == 0])
mean_weak_1 = np.mean(weak_predictions[cluster_labels == 1])
fake_cluster_index = 0 if mean_weak_0 > mean_weak_1 else 1
real_cluster_index = 1 - fake_cluster_index

baseline_preds = np.zeros(len(latent_embeddings))
baseline_preds[cluster_labels == fake_cluster_index] = 1

acc_base = accuracy_score(ground_truth, baseline_preds)
print(f"--> Immediate K-Means Unsupervised Baseline Accuracy: {acc_base:.4f}\n")

In [ ]:
print("Phase 2: The Dual-Cluster Refinement Sweep (n=250)\n")

norms = np.linalg.norm(latent_embeddings, axis=1, keepdims=True)
latent_normed = latent_embeddings / (norms + 1e-9)

def get_pristine_core(data_normed, percentile=85):
    centroid = np.mean(data_normed, axis=0, keepdims=True)
    centroid_normed = centroid / (np.linalg.norm(centroid) + 1e-9)
    dists = 1 - cosine_similarity(data_normed, centroid_normed).flatten()
    threshold = np.percentile(dists, percentile)
    return data_normed[dists <= threshold]

# 1. Prepare Core Models
X_real_cluster = latent_normed[cluster_labels == real_cluster_index]
X_fake_cluster = latent_normed[cluster_labels == fake_cluster_index]

X_real_core = get_pristine_core(X_real_cluster)
X_fake_core = get_pristine_core(X_fake_cluster)

lof_real = LocalOutlierFactor(n_neighbors=250, metric='cosine', novelty=True)
lof_real.fit(X_real_core)

lof_fake = LocalOutlierFactor(n_neighbors=250, metric='cosine', novelty=True)
lof_fake.fit(X_fake_core)

# 2. Global Anomaly Scoring
# Anomaly Score = -Decision Function. (Higher = More Anomalous)
scores_real_cluster = -lof_real.decision_function(latent_normed) # Higher means anomalous relative to REAL core
scores_fake_cluster = -lof_fake.decision_function(latent_normed) # Higher means anomalous relative to FAKE core

# 3. Grid Search Thresholds
stats_real = -lof_real.decision_function(X_real_core)
stats_fake = -lof_fake.decision_function(X_fake_core)

mean_r, std_r = np.mean(stats_real), np.std(stats_real)
mean_f, std_f = np.mean(stats_fake), np.std(stats_fake)

deviations = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]

for sd in deviations:
    thresh_r = mean_r + (sd * std_r)
    thresh_f = mean_f + (sd * std_f)
    
    final_preds = np.copy(baseline_preds)
    
    # False Negative Fix: Inside REAL cluster, but looks synthetic? -> Flip to Fake (1)
    mask_real = (cluster_labels == real_cluster_index)
    anomalies_in_real = mask_real & (scores_real_cluster > thresh_r)
    final_preds[anomalies_in_real] = 1
    
    # False Positive Fix: Inside FAKE cluster, but looks biological? -> Flip to Real (0)
    mask_fake = (cluster_labels == fake_cluster_index)
    anomalies_in_fake = mask_fake & (scores_fake_cluster > thresh_f)
    final_preds[anomalies_in_fake] = 0
    
    print("="*40)
    print(f"METRICS: DUAL FLIP AT {sd} STANDARD DEVIATIONS")
    print("="*40)
    print(f"Global Accuracy:   {accuracy_score(ground_truth, final_preds):.4f}")
    print(f"Global Precision:  {precision_score(ground_truth, final_preds, zero_division=0):.4f}")
    print(f"Global Recall:     {recall_score(ground_truth, final_preds):.4f}")
    # Composite probability for AUC (Distance to Real Core)
    print(f"Cumulative AUC:    {roc_auc_score(ground_truth, scores_real_cluster):.4f}")
    print("="*40 + "\n")